# Create Hewlett Foundation Awards (GRANT PATTERN, FacetWP method-3 via direct POST)

Ingest of the William and Flora Hewlett Foundation's public grant database at `hewlett.org/grants/database/`. ~7,000 grant cards extracted via direct POST to Hewlett's FacetWP refresh endpoint (same-origin POST to `hewlett.org/grants/`, replicated without browser automation after capturing the XHR format once via Playwright). Method #3 on the runbook ladder (search/index APIs), iterating 9 program facets to defeat FacetWP's 1000-row per-query cap.

**Awarding body:** William and Flora Hewlett Foundation — `F4320307873` (US)

**Coverage limitation (documented up front):** FacetWP caps each filtered query at 1,000 results. The full unfiltered grant total is 19,926 (verified 2026-05-20), but 5 of 9 program-filtered queries cap out at 1,000 each — meaning roughly 7,300 grants are accessible via this ingest and ~12,600 are NOT (capped programs: Education, Environment, Gender Equity and Governance, Performing Arts, Special Projects). There is no exposed year/status/alphabetic-prefix facet on the public UI to slice further. Future work to recover the missing ~12,600 grants requires either:
  - Hewlett enabling additional facets,
  - WP REST `/wp/v2/grantee` (3,162 grantee orgs) + per-grantee detail-page scrape, OR
  - Playwright drive-through of the search UI sorted by date asc/desc to bookend coverage.

Documented as a known-limitation Step-0 follow-up in the tracker row. This ingest ships what's accessible cleanly.

**Schema choices:**
- One row per (grantee × program × award_date × projectTitle) tuple. Hewlett shows one card per grant; the same grantee can receive multiple grants in different programs / years / project-titles.
- `amount` = USD value from the card's `$N,NNN.NN` text, hardcoded `currency = 'USD'` (Hewlett is US-based).
- `start_date` = the "Awarded: Month DD, YYYY" date parsed from the card meta.
- `end_date` = start_date + term_months (computed; Hewlett shows "Term: N months" which is the grant period).
- `funder_scheme` = Hewlett program name (e.g., `'Education'`, `'Environment'`, `'U.S. Democracy'`, `'Performing Arts'`, `'Racial Justice'`, etc.) — 9 distinct schemes.
- `funding_type` = `'research'` for the broad-research programs (Economy and Society, Education, Environment, U.S. Democracy, Racial Justice), `'other'` for Performing Arts + Special Projects + Effective Philanthropy + Gender Equity (some are research, some are advocacy; coarse but defensible).
- **`lead_investigator` is ORG-LEVEL, not PI-level.** Hewlett funds organizations, not individual researchers; the cards show the grantee org name but no PI. Model this like HHMI's amount-NULL waiver: `lead_investigator.given_name` / `.family_name` = NULL, `lead_investigator.affiliation.name` = grantee org. Document that the search UI doesn't expose PI data.
- `declined` boolean always False — Hewlett doesn't publish declined-application data.

**Source authority** — `hewlett.org` is the awarding body's own site (no auth, no aggregator). Same-origin FacetWP POST captured once via Playwright + replayed via plain HTTP. Method #3 on the runbook ladder.

**Prerequisites**: Run `scripts/local/hewlett_to_s3.py` first to POST through 9 programs × up to 100 pages each (~10 min wall-clock at 0.6s throttle), write parquet, upload to S3.

**Data source**: `https://hewlett.org/grants/database/` (HTTP POST to `https://hewlett.org/grants/` with FacetWP `facetwp_refresh` JSON payload)
**S3 location**: `s3a://openalex-ingest/awards/hewlett/hewlett_grants.parquet`

## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hewlett_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hewlett/hewlett_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.hewlett_raw;

## Step 1.5: Inspect raw + money/currency scan + FacetWP-cap acknowledgment

All source columns from the FacetWP card extraction. **Verified per-row coverage on the full extraction** (7,359 cards across 9 programs, validated 2026-05-20): **100% amount, 100% dates, 100% term, 100% status**. Currency = 'USD' hardcoded.

Actual extracted row count: **7,359** (sum of pre-cap totals across programs). 5 programs (Education, Environment, Gender Equity, Performing Arts, Special Projects) each cap at exactly 1000 — that ceiling means we're missing ~12,600 grants from the full 19,926 unfiltered total. **Status split: 2,244 Active + 5,115 Closed** — both card-markup variants (`ActiveGrantItem-*` and `ClosedGrantItem-*`) parsed via `(?:Active|Closed)GrantItem-{field}`. §6.7 amount-coverage **NOT waived** — expect 100%.

Source columns: `funder_award_id`, `grantee`, `program`, `facetwp_program_id`, `projectTitle`, `status`, `overview`, `amount`, `currency`, `term_months`, `start_date`, `end_date`, `amount_raw`, `term_raw`, `date_awarded_raw`, `grantee_website`, `declined`.

In [ ]:
%sql
DESCRIBE openalex.awards.hewlett_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.hewlett_raw LIMIT 5;

In [ ]:
%sql
-- §1.5 money-shape scan — confirm amount distribution looks like real Hewlett grants (~$50K to $5M typical).
SELECT
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    AVG(TRY_CAST(amount AS DOUBLE)) AS avg_amount,
    COUNT(amount) AS non_null,
    COUNT(*) AS total_rows
FROM openalex.awards.hewlett_raw;

In [ ]:
%sql
-- Currency + status sanity
SELECT currency, status, COUNT(*) AS rows FROM openalex.awards.hewlett_raw GROUP BY currency, status ORDER BY rows DESC;

## Step 1.6: Fail-fast — verify Hewlett funder row exists

In [ ]:
%sql
-- Must return exactly 1 row.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320307873;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hewlett_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320307873  -- William and Flora Hewlett Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    CONCAT('Hewlett ', n.program, ' \u2014 ', n.grantee) AS display_name,
    CASE
        WHEN n.projectTitle IS NOT NULL AND n.overview IS NOT NULL
            THEN CONCAT(n.projectTitle, '. ', n.overview)
        WHEN n.projectTitle IS NOT NULL THEN n.projectTitle
        ELSE n.overview
    END AS description,
    f.funder_id,
    n.funder_award_id,
    TRY_CAST(n.amount AS DOUBLE) AS amount,
    n.currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    -- CASE on Hewlett program. Broad research programs → 'research'; advocacy/arts → 'other'.
    -- Some judgment: Education funds both research and direct program support;
    -- defaulting to 'research' since most Education grants go to academic institutions.
    CASE
        WHEN n.program IN ('Performing Arts','Special Projects','Effective Philanthropy','Gender Equity and Governance') THEN 'other'
        ELSE 'research'
    END AS funding_type,
    n.program AS funder_scheme,
    'hewlett_facetwp' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date, 'yyyy-MM-dd') AS end_date,
    TRY_CAST(SUBSTRING(n.start_date, 1, 4) AS INT) AS start_year,
    TRY_CAST(SUBSTRING(n.end_date, 1, 4) AS INT) AS end_year,
    -- lead_investigator is ORG-LEVEL: Hewlett doesn't expose individual PIs.
    -- given_name + family_name NULL; affiliation.name = grantee org; country = 'US'
    -- (Hewlett grants are predominantly US-based; some international grantees exist
    -- but the database doesn't expose country per card).
    CASE
        WHEN n.grantee IS NULL OR n.grantee = '' THEN NULL
        ELSE struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                n.grantee AS name,
                CAST('US' AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.grantee_website AS landing_page_url,  -- grantee's own site, not Hewlett's grant detail page (Hewlett doesn't expose per-grant URLs)
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.hewlett_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.grantee IS NOT NULL;

## Step 3: Insert into openalex_awards_raw at priority 86

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hewlett_facetwp' AND priority = 86;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    86 AS priority  -- Hewlett priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.hewlett_awards;

## Step 6: Verification

Full §6.1-6.7. Amount-coverage NOT waived (every Hewlett card has $). Verified expectations from the 2026-05-20 extraction:
- Row count: **7,359** (full extraction across 9 programs)
- 100% `pct_amount` (FacetWP cards always include $)
- `currencies = [USD]`
- 9 distinct funder_schemes; 5 programs cap at exactly 1000 rows
- Status split: 2,244 Active + 5,115 Closed (30/70)
- Earliest dates likely 1990s, recent dates through 2026

In [ ]:
%sql
SELECT COUNT(*) AS total_hewlett_award_rows FROM openalex.awards.hewlett_awards;

In [ ]:
%sql
DESCRIBE openalex.awards.hewlett_awards;

In [ ]:
%sql
-- §6.3 Data completeness
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(amount) AS has_amount,
    COUNT(start_date) AS has_start_date,
    COUNT(lead_investigator) AS has_pi,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) AS pct_dates,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) AS pct_description
FROM openalex.awards.hewlett_awards;

In [ ]:
%sql
-- §6.7 amount + currency fail-fast (NOT waived).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount
FROM openalex.awards.hewlett_awards;

In [ ]:
%sql
-- Program distribution + total $ by program — useful for verifying the 1000-row FacetWP cap matches expectations.
SELECT funder_scheme AS program,
       funding_type,
       COUNT(*) AS rows,
       ROUND(SUM(amount) / 1e6, 1) AS total_usd_millions,
       ROUND(AVG(amount), 0) AS avg_amount_usd
FROM openalex.awards.hewlett_awards
GROUP BY funder_scheme, funding_type
ORDER BY rows DESC;

In [ ]:
%sql
-- Sample top-funded grants
SELECT id, SUBSTRING(display_name, 1, 80) AS title,
       funder_scheme AS program, funding_type, amount, start_year, end_year,
       SUBSTRING(lead_investigator.affiliation.name, 1, 60) AS grantee
FROM openalex.awards.hewlett_awards
WHERE amount IS NOT NULL
ORDER BY amount DESC
LIMIT 12;

In [ ]:
%sql
-- Year distribution
SELECT start_year, COUNT(*) AS rows,
       ROUND(SUM(amount)/1e6, 1) AS total_usd_millions
FROM openalex.awards.hewlett_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC
LIMIT 20;

In [ ]:
%sql
-- Funder struct populated correctly?
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.hewlett_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;